# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## Get data frame

In [22]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2025]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

In [23]:
df_train_women_list

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO
0,2010,138,3124,69,3201,55,0,22.857143,54.571429,2.285714,9.285714,11.142857,13.571429,13.857143,5.428571,14.571429,23.285714,59.857143,4.428571,14.428571,11.142857,10.714286,13.285714,6.857143,19.142857,4.428571,27.571429,66.142857,10.285714,26.428571,14.571429,14.428571,15.428571,10.857143,17.571429,25.285714,60.285714,4.000000,13.285714,13.285714,13.428571,20.857143,7.000000,16.000000,16.142857,0.500000,0.750000,4,13,-9,2047.364734
1,2010,138,3173,67,3395,66,0,24.600000,57.200000,4.800000,16.400000,13.800000,14.400000,15.000000,6.200000,18.000000,21.000000,60.400000,5.000000,14.600000,15.400000,9.600000,15.400000,6.600000,18.800000,11.400000,23.600000,58.200000,7.000000,20.200000,9.600000,15.400000,14.600000,8.000000,17.200000,22.600000,57.600000,6.000000,19.600000,13.000000,14.800000,19.000000,8.200000,16.200000,4.400000,0.500000,0.333333,8,9,-1,1803.100984
2,2010,138,3181,72,3214,37,1,24.166667,61.666667,5.833333,15.333333,17.500000,12.833333,17.166667,13.500000,19.166667,20.000000,52.833333,5.333333,16.333333,15.000000,10.000000,23.666667,8.500000,21.166667,7.333333,20.285714,52.000000,3.857143,14.714286,10.000000,11.000000,14.857143,8.000000,16.857143,16.428571,45.000000,1.428571,6.428571,10.714286,6.000000,21.285714,6.285714,16.142857,11.571429,1.000000,1.000000,2,15,-13,2222.732883
3,2010,138,3199,75,3256,61,1,25.500000,59.250000,7.750000,18.750000,14.500000,12.000000,16.750000,8.250000,17.000000,22.000000,56.000000,6.500000,19.500000,10.750000,10.500000,18.750000,7.500000,19.500000,13.000000,27.750000,63.625000,4.125000,12.875000,14.500000,14.625000,14.375000,7.125000,15.500000,25.500000,64.625000,5.500000,19.125000,14.375000,12.750000,15.875000,6.750000,17.750000,4.750000,0.000000,0.800000,3,14,-11,2065.191353
4,2010,138,3207,62,3265,42,0,24.000000,62.333333,7.333333,21.666667,16.833333,16.333333,14.166667,12.000000,15.166667,20.166667,49.666667,5.833333,21.666667,10.500000,14.166667,20.166667,5.833333,14.000000,8.333333,24.333333,55.333333,6.000000,16.000000,9.666667,13.333333,11.666667,8.500000,11.500000,20.166667,58.666667,5.833333,20.833333,14.166667,9.833333,14.500000,6.500000,16.166667,12.666667,0.500000,1.000000,5,12,-7,1901.415647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,2024,147,3425,73,3163,80,1,26.833333,67.333333,5.666667,19.500000,13.166667,13.500000,13.166667,7.166667,17.833333,24.666667,62.666667,6.333333,18.833333,6.833333,15.666667,12.333333,8.166667,19.666667,6.166667,27.833333,57.666667,7.333333,21.500000,6.166667,17.333333,11.500000,10.166667,11.833333,17.833333,56.166667,4.666667,20.666667,6.833333,10.666667,15.833333,5.166667,16.500000,29.833333,1.000000,1.000000,1,3,-2,2156.553539
1784,2024,147,3261,87,3234,94,-1,27.833333,68.000000,4.500000,13.166667,13.500000,13.666667,13.000000,8.166667,13.666667,23.166667,63.833333,4.833333,16.000000,9.000000,12.500000,15.500000,6.833333,17.833333,15.500000,34.666667,67.833333,14.833333,34.500000,9.666667,25.666667,13.833333,7.333333,13.333333,27.666667,70.000000,9.833333,27.500000,9.833333,17.000000,13.333333,7.500000,18.666667,23.166667,0.666667,1.000000,3,1,2,2242.701954
1785,2024,151,3163,69,3234,71,0,27.833333,57.666667,7.333333,21.500000,6.166667,17.333333,11.500000,10.166667,11.833333,17.833333,56.166667,4.666667,20.666667,6.

In [24]:
x_train_women

array([[ 2.28571429e+01,  5.45714286e+01,  2.28571429e+00, ...,
         1.30000000e+01, -9.00000000e+00,  2.04736473e+03],
       [ 2.46000000e+01,  5.72000000e+01,  4.80000000e+00, ...,
         9.00000000e+00, -1.00000000e+00,  1.80310098e+03],
       [ 2.41666667e+01,  6.16666667e+01,  5.83333333e+00, ...,
         1.50000000e+01, -1.30000000e+01,  2.22273288e+03],
       ...,
       [ 2.78333333e+01,  5.76666667e+01,  7.33333333e+00, ...,
         1.00000000e+00,  2.00000000e+00,  2.33790587e+03],
       [ 2.40000000e+01,  6.15000000e+01,  3.66666667e+00, ...,
         1.00000000e+00,  2.00000000e+00,  2.21182212e+03],
       [ 3.46666667e+01,  6.78333333e+01,  1.48333333e+01, ...,
         1.00000000e+00,  0.00000000e+00,  2.30663400e+03]])

In [25]:
df_train_men_list

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO,CoachELO
0,2010,134,1115,61,1457,44,0,18.750000,47.500000,4.375000,14.875000,11.375000,12.500000,16.125000,6.625000,23.375000,16.625000,48.875000,3.750000,13.625000,9.625000,9.125000,13.625000,8.125000,22.375000,6.000000,22.857143,57.571429,4.000000,14.714286,11.571429,11.714286,11.428571,8.000000,19.000000,21.714286,53.714286,4.428571,17.857143,12.142857,11.142857,15.000000,5.714286,17.000000,1.428571,0.800000,1.000000,16,16,0,1293.620328,1529.250804
1,2010,136,1124,68,1358,59,0,27.571429,55.857143,6.571429,15.857143,11.000000,13.000000,13.714286,7.142857,18.857143,24.714286,59.571429,6.285714,18.714286,13.571429,13.142857,11.571429,6.857143,19.857143,7.000000,27.875000,60.125000,9.750000,25.875000,13.250000,19.375000,13.500000,7.500000,22.875000,24.875000,58.625000,5.625000,19.000000,12.000000,14.875000,14.000000,7.000000,22.750000,11.250000,0.750000,0.800000,3,14,-11,1936.132062,1982.513009
2,2010,136,1139,77,1431,59,0,23.250000,47.500000,7.750000,19.000000,6.000000,12.250000,13.000000,6.000000,18.750000,20.500000,56.250000,3.500000,20.500000,11.000000,9.500000,12.250000,6.500000,21.500000,14.000000,26.375000,56.125000,5.375000,15.250000,9.125000,12.625000,13.625000,8.375000,18.625000,21.875000,56.125000,6.500000,19.500000,13.125000,12.500000,14.375000,6.125000,17.250000,7.375000,1.000000,0.800000,5,12,-7,1961.585673,2016.224664
3,2010,136,1140,99,1196,92,0,27.428571,59.571429,8.000000,19.571429,9.285714,14.142857,9.714286,8.714286,17.714286,25.000000,55.285714,7.714286,20.142857,7.428571,13.428571,16.142857,5.142857,19.571429,12.857143,24.857143,57.428571,5.571429,16.714286,13.285714,12.285714,11.000000,7.000000,12.428571,25.714286,54.714286,7.857143,19.857143,10.000000,13.142857,13.285714,5.428571,17.428571,0.714286,0.750000,0.250000,7,10,-3,1914.299666,1970.388403
4,2010,136,1242,90,1250,74,0,27.875000,55.625000,7.000000,15.250000,11.250000,16.125000,13.625000,7.750000,17.750000,24.875000,59.250000,6.250000,19.000000,11.625000,12.625000,12.375000,7.500000,20.125000,12.000000,26.500000,52.666667,7.666667,17.333333,9.333333,16.833333,12.333333,7.500000,15.666667,24.333333,55.333333,6.833333,19.833333,9.000000,14.000000,13.666667,6.500000,19.166667,12.333333,1.000000,1.000000,1,16,-15,2163.740204,2204.083246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1883,2024,146,1181,64,1301,76,0,29.000000,60.000000,8.500000,23.166667,10.666667,14.833333,8.833333,6.166667,16.500000,25.833333,56.333333,5.666667,15.166667,8.166667,12.333333,9.333333,5.333333,12.000000,8.500000,27.400000,58.400000,6.700000,17.800000,8.800000,12.300000,9.500000,6.500000,13.600000,28.800000,61.700000,7.500000,21.500000,8.200000,14.900000,10.100000,5.800000,16.400000,2.000000,0.333333,0.714286,4,11,-7,2041.214285,2027.939972
1884,2024,146,1397,66,1345,72,0,25.333333,62.666667,9.166667,27.666667,11.166667,15.500000,9.666667,7.333333,17.666667,24.000000,58.500000,8.666667,27.333333,8.500000,13.500000,12.166667,6.666667,19.333333,6.000000,26.166667,55.666667,7.500000,18.000000,10.000000,18.500000,10.166667,5.166667,16.666667,27.000000,62.666667,6.333333,22.000000,9.666667,12.500000,9.166667,6.833333,23.000000,5.333333,0.333333,0.750000,2,1,1,2037.027527,2086.840863
1885,2024,152,1104,72,1163,86,0,31.500000,67.666667,8.333333,28.166667,9.833333

In [26]:
x_train_men

array([[  18.75      ,   47.5       ,    4.375     , ...,    0.        ,
        1293.62032833, 1529.25080433],
       [  27.57142857,   55.85714286,    6.57142857, ...,  -11.        ,
        1936.13206217, 1982.51300945],
       [  23.25      ,   47.5       ,    7.75      , ...,   -7.        ,
        1961.58567329, 2016.22466397],
       ...,
       [  31.5       ,   67.66666667,    8.33333333, ...,    3.        ,
        2018.94042036, 2068.95188099],
       [  27.4       ,   58.4       ,    6.7       , ...,   10.        ,
        1954.78353941, 1981.34106167],
       [  26.16666667,   55.66666667,    7.5       , ...,    0.        ,
        2178.11347661, 2191.54686187]])

## Save x

In [27]:
x_women_DF = pd.DataFrame(x_train_women) 

In [28]:
x_women_DF

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43
0,22.857143,54.571429,2.285714,9.285714,11.142857,13.571429,13.857143,5.428571,14.571429,23.285714,59.857143,4.428571,14.428571,11.142857,10.714286,13.285714,6.857143,19.142857,4.428571,27.571429,66.142857,10.285714,26.428571,14.571429,14.428571,15.428571,10.857143,17.571429,25.285714,60.285714,4.000000,13.285714,13.285714,13.428571,20.857143,7.000000,16.000000,16.142857,0.500000,0.750000,4.0,13.0,-9.0,2047.364734
1,24.600000,57.200000,4.800000,16.400000,13.800000,14.400000,15.000000,6.200000,18.000000,21.000000,60.400000,5.000000,14.600000,15.400000,9.600000,15.400000,6.600000,18.800000,11.400000,23.600000,58.200000,7.000000,20.200000,9.600000,15.400000,14.600000,8.000000,17.200000,22.600000,57.600000,6.000000,19.600000,13.000000,14.800000,19.000000,8.200000,16.200000,4.400000,0.500000,0.333333,8.0,9.0,-1.0,1803.100984
2,24.166667,61.666667,5.833333,15.333333,17.500000,12.833333,17.166667,13.500000,19.166667,20.000000,52.833333,5.333333,16.333333,15.000000,10.000000,23.666667,8.500000,21.166667,7.333333,20.285714,52.000000,3.857143,14.714286,10.000000,11.000000,14.857143,8.000000,16.857143,16.428571,45.000000,1.428571,6.428571,10.714286,6.000000,21.285714,6.285714,16.142857,11.571429,1.000000,1.000000,2.0,15.0,-13.0,2222.732883
3,25.500000,59.250000,7.750000,18.750000,14.500000,12.000000,16.750000,8.250000,17.000000,22.000000,56.000000,6.500000,19.500000,10.750000,10.500000,18.750000,7.500000,19.500000,13.000000,27.750000,63.625000,4.125000,12.875000,14.500000,14.625000,14.375000,7.125000,15.500000,25.500000,64.625000,5.500000,19.125000,14.375000,12.750000,15.875000,6.750000,17.750000,4.750000,0.000000,0.800000,3.0,14.0,-11.0,2065.191353
4,24.000000,62.333333,7.333333,21.666667,16.833333,16.333333,14.166667,12.000000,15.166667,20.166667,49.666667,5.833333,21.666667,10.500000,14.166667,20.166667,5.833333,14.000000,8.333333,24.333333,55.333333,6.000000,16.000000,9.666667,13.333333,11.666667,8.500000,11.500000,20.166667,58.666667,5.833333,20.833333,14.166667,9.833333,14.500000,6.500000,16.166667,12.666667,0.500000,1.000000,5.0,12.0,-7.0,1901.415647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,26.833333,67.333333,5.666667,19.500000,13.166667,13.500000,13.166667,7.166667,17.833333,24.666667,62.666667,6.333333,18.833333,6.833333,15.666667,12.333333,8.166667,19.666667,6.166667,27.833333,57.666667,7.333333,21.500000,6.166667,17.333333,11.500000,10.166667,11.833333,17.833333,56.166667,4.666667,20.666667,6.833333,10.666667,15.833333,5.166667,16.500000,29.833333,1.000000,1.000000,1.0,3.0,-2.0,2156.553539
1784,27.833333,68.000000,4.500000,13.166667,13.500000,13.666667,13.000000,8.166667,13.666667,23.166667,63.833333,4.833333,16.000000,9.000000,12.500000,15.500000,6.833333,17.833333,15.500000,34.666667,67.833333,14.833333,34.500000,9.666667,25.666667,13.833333,7.333333,13.333333,27.666667,70.000000,9.833333,27.500000,9.833333,17.000000,13.333333,7.500000,18.666667,23.166667,0.666667,1.000000,3.0,1.0,2.0,2242.701954
1785,27.833333,57.666667,7.333333,21.500000,6.166667,17.333333,11.500000,10.166667,11.833333,17.833333,56.166667,4.666667,20.666667,6.833333,10.666667,15.833333,5.166667,16.500000,29.833333,34.666667,67.833333,14.833333,34.500000,9.666667,25.666667,13.833333,7.333333,13.333333,27.666667,70.000000,9.833333,27.500000,9.833333,17.000000,13.333333,7.500000,18.666667,23.166667,1.000000,1.000000,3.0,1.0,2.0,2337.905869
1786,24.000000,61.500000,3.666667,15.500000,9.333333,11.500000,12.166667,5.000000,11.500000,22.500000,65.666667,5.833333,22.000000,8.666667,9.000000,10.166667,6.666667,15.333333,6.000000,32.000000,66.166667,5.833333,15.000000,13.000000,19.833333,13.666667,7.000000,16.500000,23.000000,63.333333,6.000000,21.833333,9.500000,11.500000,13.500000,5.666667,16.333333,1

In [31]:
x_women_DF.to_csv("WomenData_2010_2024.csv")

In [29]:
x_men_DF = pd.DataFrame(x_train_men) 

In [30]:
x_men_DF

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44
0,18.750000,47.500000,4.375000,14.875000,11.375000,12.500000,16.125000,6.625000,23.375000,16.625000,48.875000,3.750000,13.625000,9.625000,9.125000,13.625000,8.125000,22.375000,6.000000,22.857143,57.571429,4.000000,14.714286,11.571429,11.714286,11.428571,8.000000,19.000000,21.714286,53.714286,4.428571,17.857143,12.142857,11.142857,15.000000,5.714286,17.000000,1.428571,0.800000,1.000000,16.0,16.0,0.0,1293.620328,1529.250804
1,27.571429,55.857143,6.571429,15.857143,11.000000,13.000000,13.714286,7.142857,18.857143,24.714286,59.571429,6.285714,18.714286,13.571429,13.142857,11.571429,6.857143,19.857143,7.000000,27.875000,60.125000,9.750000,25.875000,13.250000,19.375000,13.500000,7.500000,22.875000,24.875000,58.625000,5.625000,19.000000,12.000000,14.875000,14.000000,7.000000,22.750000,11.250000,0.750000,0.800000,3.0,14.0,-11.0,1936.132062,1982.513009
2,23.250000,47.500000,7.750000,19.000000,6.000000,12.250000,13.000000,6.000000,18.750000,20.500000,56.250000,3.500000,20.500000,11.000000,9.500000,12.250000,6.500000,21.500000,14.000000,26.375000,56.125000,5.375000,15.250000,9.125000,12.625000,13.625000,8.375000,18.625000,21.875000,56.125000,6.500000,19.500000,13.125000,12.500000,14.375000,6.125000,17.250000,7.375000,1.000000,0.800000,5.0,12.0,-7.0,1961.585673,2016.224664
3,27.428571,59.571429,8.000000,19.571429,9.285714,14.142857,9.714286,8.714286,17.714286,25.000000,55.285714,7.714286,20.142857,7.428571,13.428571,16.142857,5.142857,19.571429,12.857143,24.857143,57.428571,5.571429,16.714286,13.285714,12.285714,11.000000,7.000000,12.428571,25.714286,54.714286,7.857143,19.857143,10.000000,13.142857,13.285714,5.428571,17.428571,0.714286,0.750000,0.250000,7.0,10.0,-3.0,1914.299666,1970.388403
4,27.875000,55.625000,7.000000,15.250000,11.250000,16.125000,13.625000,7.750000,17.750000,24.875000,59.250000,6.250000,19.000000,11.625000,12.625000,12.375000,7.500000,20.125000,12.000000,26.500000,52.666667,7.666667,17.333333,9.333333,16.833333,12.333333,7.500000,15.666667,24.333333,55.333333,6.833333,19.833333,9.000000,14.000000,13.666667,6.500000,19.166667,12.333333,1.000000,1.000000,1.0,16.0,-15.0,2163.740204,2204.083246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1883,29.000000,60.000000,8.500000,23.166667,10.666667,14.833333,8.833333,6.166667,16.500000,25.833333,56.333333,5.666667,15.166667,8.166667,12.333333,9.333333,5.333333,12.000000,8.500000,27.400000,58.400000,6.700000,17.800000,8.800000,12.300000,9.500000,6.500000,13.600000,28.800000,61.700000,7.500000,21.500000,8.200000,14.900000,10.100000,5.800000,16.400000,2.000000,0.333333,0.714286,4.0,11.0,-7.0,2041.214285,2027.939972
1884,25.333333,62.666667,9.166667,27.666667,11.166667,15.500000,9.666667,7.333333,17.666667,24.000000,58.500000,8.666667,27.333333,8.500000,13.500000,12.166667,6.666667,19.333333,6.000000,26.166667,55.666667,7.500000,18.000000,10.000000,18.500000,10.166667,5.166667,16.666667,27.000000,62.666667,6.333333,22.000000,9.666667,12.500000,9.166667,6.833333,23.000000,5.333333,0.333333,0.750000,2.0,1.0,1.0,2037.027527,2086.840863
1885,31.500000,67.666667,8.333333,28.166667,9.833333,15.833333,11.166667,5.666667,23.333333,30.666667,62.833333,8.500000,24.166667,8.333333,14.166667,10.000000,7.000000,21.000000,-7.000000,29.285714,59.428571,9.285714,24.285714,9.285714,21.000000,7.857143,5.000000,14.285714,23.000000,59.857143,6.142857,20.571429,7.571429,9.142857,8.428571,5.428571,14.428571,17.571429,0.333333,1.000000,4.0,1.0,3.0,2018.940420,2068.951881
1886,27.400000,58.400000,6.700000,17.800000,8.800000,12.300000,9.500000,6.500000,13.600000,28.800000,61.700000,7.500000,21.500000,8.200000,14.900000,10.100000,5.800000,16.400000,2.000000,26.166667,55.666667,7.500000,18.000000,10.000000,18.500000,10.166667,5.166667,16.666667,27.0

In [32]:
x_men_DF.to_csv("MenData_2010_2024.csv")